### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.ticker import MultipleLocator

import ADT_OT2_Analysis as AD

### Variables

Membrane Thickness:

| Sample | Before (mm) | After (mm) |
| --- | --- | --- |
| S1 |  |  |
| S2 |  |  |
| S3 |  |  |
| S4 |  |  |

In [2]:
# Testing Details

Date = 'YYYYMMDD'
Membrane = 'membrane'
Dye_name = 'Dye_name'
dye_sh = 'dye_shorthand'
dilution_factor = 10

# Membrane sample names (in order of H1, H2, H3, H4)
# membrane_L is membrane thickness (in microns). Default is 110
# total_volume is the total volume of solution in the H-cell (mL)
H_cells = { }
 
H_cells['H1'] = {'sample':'S1', 'membrane_L': 110, 'total_volume' : 22}
H_cells['H2'] = {'sample':'S2', 'membrane_L': 110, 'total_volume' : 22}
H_cells['H3'] = {'sample':'S3', 'membrane_L': 110, 'total_volume' : 22}
H_cells['H4'] = {'sample':'S4', 'membrane_L': 110, 'total_volume' : 22}

#H_cells['H5'] = {'sample':'S5', 'membrane_L': 110, 'total_volume' : 22}
#H_cells['H6'] = {'sample':'S6', 'membrane_L': 110, 'total_volume' : 22}
#H_cells['H7'] = {'sample':'S7', 'membrane_L': 110, 'total_volume' : 22}
#H_cells['H8'] = {'sample':'S8', 'membrane_L': 110, 'total_volume' : 22}

In [3]:
#experiment_handle = '20250412_ot2_PES_P40G00_RB'
experiment_handle = Date + '_ot2_PES_' + Membrane + '_' + dye_sh

# Files to read
calibration_curve_csv = 'OT2_Data/'+ Date + '_ot2_calibration_' + dye_sh + '.csv'
longform_filename = 'OT2_Data/' + experiment_handle + '_long.csv'
shortform_filename = 'OT2_Data/' + experiment_handle + '_short.csv'

# What you want to name the files that will be saved
longform_save_name = 'Results/' + experiment_handle + '_metadata.csv'
full_df_save_name ='Results/' + experiment_handle + '.csv'

In [3]:
# Labels for plot legends
plot_H1 = Membrane + ' ' + H_cells['H1']['sample']
plot_H2 = Membrane + ' ' + H_cells['H2']['sample']
plot_H3 = Membrane + ' ' + H_cells['H3']['sample']
plot_H4 = Membrane + ' ' + H_cells['H4']['sample']

#plot_H5 = Membrane + ' ' + H_cells['H5']['sample']
#plot_H6 = Membrane + ' ' + H_cells['H6']['sample']
#plot_H7 = Membrane + ' ' + H_cells['H7']['sample']
#plot_H8 = Membrane + ' ' + H_cells['H8']['sample']

# Adjust as needed
plot_labels = [plot_H1, plot_H2, plot_H3, plot_H4]

# Colors by dye type
rb = 'palevioletred'
mb = 'rebeccapurple'
bb = 'royalblue'
ao = 'darkorange'
contrast = 'dimgrey'
DT_suite = [rb, mb, bb, ao]

dye = bb

# comparison options
contrast_A = 'cornflowerblue'
contrast_B = 'forestgreen'
contrast_C = 'violet'

#colorblind suite
color_A = (0,0.306,0.920) #blue
color_B = (0,0.5,0) #green
color_C = (0.73,0.33,0.83) #fuschia
color_D = (0.88,0.55,0) #orange
CB_suite = [color_A, color_B, color_C, color_D]

# Original suite
OG_A = 'cornflowerblue'
OG_B = 'forestgreen'
OG_C = 'peru'
OG_D = 'palevioletred'
OG_suite = [OG_A, OG_B, OG_C, OG_D]

## Calibration Curve

typical abs slope per dye:

| Dye | Abs | or |
| --- | --- | --- |
| BB | 0.069 | 0.0723 | 
| AO | 0.0135 |  |
| RB | 0.0589 | 0.058 |
| MB | 0.0327 | 0.035 |

In [ ]:
#absorption coefficient dertermined from Calibration section
m, b = AD.absorptivity(calibration_curve_csv)
absorptivity = round(m,4)

print('absorptivity = ', absorptivity)

In [ ]:
# plotting the calibration curve for a sanity check (optional)
calibration_curve_df = pd.read_csv(calibration_curve_csv)
X = calibration_curve_df['C_uM']
Y = calibration_curve_df['Abs']
yfit = [b + m * xi for xi in X]

fig, ax = plt.subplots(figsize=(3.5,3.5))

ax.plot(X, Y, ms=6, marker='o', linewidth=0, color='black')
ax.plot(X, yfit, linewidth=2, color = 'darkorange')
ax.set_xlabel(r'Concentration ($\mu M$)', fontsize=12)
ax.set_ylabel('Abs', fontsize=12)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show();

## Analysis

### Part 1: Longform .csv

In [ ]:
longform = AD.longform_record(longform_filename, absorptivity, dilution_factor)
longform

### Save the metadata file:

In [ ]:
longform.to_csv(longform_save_name, index=False)

### Part 2: Shortform .csv (Concentrations, Averages, and Diffusion Coefficients)
This is where the shortform file is converted to a new df with the original dye concentrations (instead of UV-Vis absorbances)

The concentration data for each sample is then used to determine:
- Diffusion and Permeability coefficients calculated at each time point per membrane sample
- overall diffusion and permeability coefficient for each individual sample, averaged over the given index range
- default inputs:
    - absorptivity: calibration curve slope, calculated above
    - dilution_factor: dilution factor of the samples, as defined in the first cell
    - H_cells: metadata about each H_cell sample, as defined in the first cell
- adjustable inputs:
    - radius : radius of the exposed area of the membrane, in centimeters
    - index_range : selection of data to calculate overall diffusivity/permeability

In [ ]:
full_df, df_aves = AD.data_calculations(shortform_filename, absorptivity, dilution_factor,
                                        radius=0.8, H_cells=H_cells,
                                        index_range=[1,2,3,4,5,6,7,8,9,10,11])
full_df

In [ ]:
df_aves

### Plots

Check the progress of the experiment using D or P calculated values (`'D'`, `'P'`)

In [ ]:
AD.progress_plots(full_df, 'P', sample_names=H_cells, color_set=OG_suite, y_subticks=0.5, markers=6, font=12)

If there are issues with specific samples (i.e. the pipette dropped a tip so you don't want to include the UV-Vis measurement for that sample/iteration), use the following to select which data to be included in the overall P/D calculation:

`AD.data_averages_1_4` is specific to H-cells labeled H1 to H4. 

`AD.data_averages_5_8` is specific to H-cells labeled H5 to H8.

If using all 8 H-cells, use `AD.data_averages_8` instead.

In [ ]:
AD.data_averages_1_4(full_df, H_cells, 'P',
                     H1_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H2_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H3_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H4_index=[1,2,3,4,5,6,7,8,9,10,11])

In [ ]:
AD.data_averages_1_4(full_df, H_cells, 'D',
                     H1_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H2_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H3_index=[1,2,3,4,5,6,7,8,9,10,11],
                     H4_index=[1,2,3,4,5,6,7,8,9,10,11])

### Save the final DF

Save to the results folder

In [ ]:
full_df.to_csv(full_df_save_name, index=False)

### Tracking Ctot

To see if the concentration in the H-cells are staying consistent throughout the experiment, add the C1 and C2 columns from each H-cell together:

In [ ]:
sum_df = pd.DataFrame(data=full_df)
sum_df['H1_sum'] = sum_df['H1_C1']+sum_df['H1_C2']
sum_df['H2_sum'] = sum_df['H2_C1']+sum_df['H2_C2']
sum_df['H3_sum'] = sum_df['H3_C1']+sum_df['H3_C2']
sum_df['H4_sum'] = sum_df['H4_C1']+sum_df['H4_C2']

sum_df[['H1_sum','H2_sum','H3_sum','H4_sum']]

## Other Plots

### Total Concentration vs Time

Keep track of overall concentration of the dye in each H-cell over time (in case of partitioning/evaporation/non-PSS diffusion behavior)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3.5))

ax.plot(sum_df['Time'], sum_df['H1_sum'], label=plot_H1, ms=8, marker='o',
        color=CB_suite[0], linewidth=0)
ax.plot(sum_df['Time'], sum_df['H2_sum'], label=plot_H2, ms=8, marker='o',
        color=CB_suite[1], linewidth=0)
ax.plot(sum_df['Time'], sum_df['H3_sum'], label=plot_H3, ms=8, marker='o',
        color=CB_suite[2], linewidth=0)
ax.plot(sum_df['Time'], sum_df['H4_sum'], label=plot_H4, ms=8, marker='o',
        color=CB_suite[3], linewidth=0)
ax.set_xlabel('Time (hr)', fontsize=12)
ax.set_ylabel(r'Concentration ($\mu M$)', fontsize=12)
ax.set_ylim(sum_df['H1_sum'].min()-100,sum_df['H1_sum'].max()+50)
ax.set_xlim(-2,sum_df['Time'].iloc[-1]+5)
ax.legend(loc='lower center', ncols=2, fontsize=9, edgecolor='inherit')
ax.tick_params(labelsize=12)

plt.tight_layout()
#plt.savefig('../Figures/'+ Date + 'fig_name.csv', dpi=300)
plt.show();

### Bar Chart

Bar chart of each individual membrane sample in this round of experiments.

In [ ]:
samples = (H_cells['H1']['sample'], H_cells['H2']['sample'], H_cells['H3']['sample'], H_cells['H4']['sample'])
ind = np.arange(len(samples)) #the x locations of the groups
bar_width = 0.25 # the width of the bars

# Choose 'P' or 'D'
VarA = '_'+'P'

B1 = H_cells['H1']['sample']
B2 = H_cells['H2']['sample']
B3 = H_cells['H3']['sample']
B4 = H_cells['H4']['sample']

B1_index = full_df[B1+ VarA][1:]
B2_index = full_df[B2+ VarA][1:]
B3_index = full_df[B3+ VarA][1:]
B4_index = full_df[B4+ VarA][1:]

bar_data = [B1_index.mean(),B2_index.mean(),B3_index.mean(),B4_index.mean()]
bar_data_std = [B1_index.std(),B2_index.std(),B3_index.std(),B4_index.std()]

# colors by dye type
bb_A = '#1b27a5'
bb_B = '#65b5fc'

ao_A = '#a65404'
ao_A = '#f5ae6a'

rb_A = '#9e1769'
rb_B = '#eb92c8'

mb_A = '#663399'
mb_A = '#e5c8ef'

fig1, ax1 = plt.subplots(figsize=(6,3), layout='constrained')

ax1.bar(ind, bar_data, bar_width, yerr=bar_data_std, label='Permeability', capsize=2, color=rb_A)

ax1.set_xticks(ind, labels=samples, fontsize=11, wrap=True)
ax1.set_ylabel('(cm/s) $ (x10^-5$)', fontsize=12)
ax1.tick_params(labelsize=12)

ax1.set_title(dye_sh + '-' + Membrane, fontsize=12)
ax1.legend(loc='upper right', ncols=1, fontsize=12)
ax1.set_ylim(0, 6)
ax1.yaxis.set_minor_locator(MultipleLocator(10))

#plt.savefig('', dpi=300)
plt.show()

### Comparison Plot

Compare two (or more) samples from different experiments

In [1]:
df1 = pd.read_csv('Results/20250603_manual_PES_P25G00_AO.csv')
df2 = pd.read_csv('Results/20250505_ot2_PES_P25G00_AO.csv')

fig2, (ax2,ax3) = plt.subplots(nrows=1, ncols=2, figsize=(8,4), layout='constrained')
#fig2.suptitle('Manual vs Automated Sampling', fontsize=20)

donor = mlines.Line2D([], [], color='black', ls='', marker='v',
                          markersize=8, label='Donor Chamber')
receptor = mlines.Line2D([], [], color='black', ls='', marker='s',
                          markersize=8, label='Receptor Chamber')
diffusivity = mlines.Line2D([], [], color='black', ls='', marker='o',
                          markersize=8, label='Diffusion Coefficient')

sample_1 = mpatches.Patch(color='#332288', label='By Hand')
sample_2 = mpatches.Patch(color='#44AA99', label='Automated')


ax2.plot(df2['Time'], df2['H3_C1']/df2['H3_C1'][0], label='Automated (Donor)', ms=9, 
         marker='v', color='#44AA99', linewidth=0)
ax2.plot(df2['Time'], df2['H3_C2']/df2['H3_C1'][0], label='Automated (Receptor)', ms=9, 
         marker='s', color='#44AA99', linewidth=0)
ax2.plot(df1['Time'], df1['H1_C1']/df1['H1_C1'][0], label='By Hand (Donor)',
         ms=9, marker='v', color='#332288', linewidth=0)
ax2.plot(df1['Time'], df1['H1_C2']/df1['H1_C1'][0], label='By Hand (Receptor)',
         ms=9, marker='s', color='#332288', linewidth=0)

ax2.set_xlabel('Time (hr)', fontsize=12)
ax2.set_ylabel('RB Concentration (normalized)', fontsize=12)
ax2.set_ylim(0,1.1)
ax2.set_xlim(-2,70)
ax2.xaxis.set_minor_locator(MultipleLocator(2))
ax2.legend(handles=[donor, receptor], fontsize=12, edgecolor='inherit')
ax2.tick_params(labelsize=12)


ax3.plot(df2['Time'], df2['S3_D'], label='Automated (PEG25)', ms=10, marker='o', color='#44AA99', linewidth=0)
ax3.plot(df1['Time'], df1['S1_D'], label='By Hand (PEG25)', ms=10, marker='o', color='#332288', linewidth=0)
ax3.set_xlabel('Time (hr)', fontsize=12)
ax3.set_ylabel('Diffusivity ($\mu m^2/s$)', fontsize=12)
ax3.set_ylim(0,60)
ax3.set_xlim(0,70)
ax3.xaxis.set_minor_locator(MultipleLocator(2))
ax3.legend(handles=[diffusivity], fontsize=12, edgecolor='inherit')
ax3.tick_params(labelsize=12)

fig2.legend(handles=[sample_1, sample_2], loc='outside lower center', ncols=4,
           fontsize=12, edgecolor='inherit')

plt.show();

NameError: name 'pd' is not defined